# Estudio 06 · Fase 6 · Enmienda 6a′ (GLM Poisson con la α de la NB2)

Enmienda `PHASE6_AMENDMENT` (commit `e37e55c`), pedida por Rosa **después de ver** que el GLM NB2 de 6a no
convergió. Es la única enmienda de la fase 6. 6a′ = GLM Poisson por horizonte (IRLS) con offset
log(hist_lag_a + 0,5), las 9 columnas de C3 estandarizadas por bloque y la α de la NB2 con el estimador de L. Un
ajuste que no converge se reajusta con L2 = 1e-4 (sin penalizar el intercepto) y se reporta. Si no converge sin
penalización en la mayoría de los 54 ajustes (bloque × horizonte), el GLM queda como no interpretable.


In [1]:
import json
import sys

import numpy as np
import pandas as pd
import statsmodels

from denguekay.config.paths import RESEARCH
from denguekay.evaluation import episodes as diag
from denguekay.model import tracking

STUDY = RESEARCH / "06_model_improvement"
sys.path.insert(0, str(STUDY / "code"))
import phase3_onsets as p3
import phase5_counts as p5
import phase6_other_families as p6
import preregistration as pr

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 300)
METRICS = STUDY / "docs" / "metricas" / "fase6.json"
FAMILY = "6a_prima_poisson_glm"
inp = p3.phase3_inputs()
statsmodels.__version__


'0.15.0'

## 1 · Guardas: reproducción de L + 5a y G14


In [2]:
base = p6.l_plus_5a(inp["serie"])
saved5 = json.loads((STUDY / "docs" / "metricas" / "fase5.json").read_text(encoding="utf-8"))
g16_repro = p6.guard_g16_reproduction(base["evaluaciones"], saved5["veredictos"]["5a"]["metricas"])
run = p6.run_family(inp, FAMILY, pr.SEEDS)
g14 = p6.guard_g14(run["ajustes"], p5.read_l_alphas())
if not (g16_repro["pasa"] and g14["pasa"]):
    raise RuntimeError(f"Falla una guarda: se detiene la fase. G14={g14} G16={g16_repro}")
{"G14": g14, "G16_reproduccion": g16_repro}


{'G14': {'mismos_bloques': True, 'bloques_distintos': 0, 'pasa': True},
 'G16_reproduccion': {'diferencia_maxima': 4.9451664985156185e-11,
  'filas': 18,
  'pasa': True}}

## 2 · Convergencia y regla de cierre


In [3]:
fits = run["ajustes"]
conv_cols = [f"convergio_h{h}" for h in p6.H]
pen_cols = [f"penalizado_h{h}" for h in p6.H]
converged = int(fits[conv_cols].to_numpy().sum())
total = fits[conv_cols].size
interpretable = converged > total / 2
print(f"Ajustes que convergen sin penalización: {converged} de {total}; interpretable: {interpretable}")
fits[["bloque", "filas_entrenamiento", *conv_cols, *pen_cols]]


Ajustes que convergen sin penalización: 54 de 54; interpretable: True


,bloque,filas_entrenamiento,convergio_h2,convergio_h3,convergio_h4,penalizado_h2,penalizado_h3,penalizado_h4
0,temporada_2008,8645,True,True,True,False,False,False
1,temporada_2009,12025,True,True,True,False,False,False
2,temporada_2010,15470,True,True,True,False,False,False
3,temporada_2011,18850,True,True,True,False,False,False
4,temporada_2012,22230,True,True,True,False,False,False
5,temporada_2013,25610,True,True,True,False,False,False
6,temporada_2014,28990,True,True,True,False,False,False
7,temporada_2015,32370,True,True,True,False,False,False
8,temporada_2016,35815,True,True,True,False,False,False
9,temporada_2017,39195,True,True,True,False,False,False


In [4]:
p6.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
pd.concat(run["predicciones"].values(), ignore_index=True).to_csv(p6.OUTPUT_DIR / f"predicciones_{FAMILY}.csv.gz", index=False)
fits.drop(columns=["corte"]).to_csv(p6.OUTPUT_DIR / f"ajustes_{FAMILY}.csv", index=False)


## 3 · Lectura frente a L + 5a (solo si es interpretable se usa en el informe)


In [5]:
f1_rule = p3.compare(run, base)
mae_rule = p5.compare(run, base)
mae_rule_mu = p5.compare({"evaluaciones": run["evaluaciones_mu"]}, base)
pd.concat({r: d["veredicto"][["horizonte", "veredicto", "vetos"]] for r, d in
           (("F1 (fase 3)", f1_rule), ("razón MAE (fase 5)", mae_rule), ("razón MAE, μ sin mezclar", mae_rule_mu))},
          names=["regla"]).reset_index(level=0)


,regla,horizonte,veredicto,vetos
0,F1 (fase 3),2,empate,brier
1,F1 (fase 3),3,empate,brier
2,F1 (fase 3),4,gana,NaN
0,razón MAE (fase 5),2,empate,brier
1,razón MAE (fase 5),3,empate,brier
2,razón MAE (fase 5),4,empate,NaN
0,"razón MAE, μ sin mezclar",2,empate,"razon_rmse, brier"
1,"razón MAE, μ sin mezclar",3,empate,"razon_rmse, brier"
2,"razón MAE, μ sin mezclar",4,empate,razon_rmse


In [6]:
f1_rule["antelacion"].round(3)


,horizonte,episodios,diferencia,ic95_inferior,ic95_superior,diferencia_temporada_2010,diferencia_temporada_2015,diferencia_temporada_2016,diferencia_temporada_2022,diferencia_temporada_2023,episodios_temporada_2010,episodios_temporada_2015,episodios_temporada_2016,episodios_temporada_2022,episodios_temporada_2023
0,2,193,0.057,0.021,0.098,0.033,0.029,0.147,0.000,0.070,30,34,34,38,57
1,3,193,0.078,0.041,0.119,0.033,0.059,0.235,0.026,0.053,30,34,34,38,57
2,4,193,0.119,0.073,0.166,0.000,0.088,0.382,0.053,0.088,30,34,34,38,57


In [7]:
cols = ["horizonte", "bloque", "positivos_cand", "f1_050_cand", "f1_050_ref", "auprc_cand", "auprc_ref", "brier_cand",
        "brier_ref"]
f1_rule["metricas"][cols].round(3)


,horizonte,bloque,positivos_cand,f1_050_cand,f1_050_ref,auprc_cand,auprc_ref,brier_cand,brier_ref
3,2,temporada_2010,234.0,0.733,0.722,0.788,0.780,0.031,0.030
8,2,temporada_2015,557.0,0.785,0.772,0.886,0.885,0.050,0.051
9,2,temporada_2016,395.0,0.652,0.657,0.750,0.757,0.064,0.059
15,2,temporada_2022,431.0,0.706,0.712,0.779,0.781,0.054,0.054
16,2,temporada_2023,1039.0,0.860,0.857,0.948,0.947,0.062,0.062
17,2,temporada_2024,857.0,0.790,0.799,0.890,0.893,0.076,0.074
21,3,temporada_2010,234.0,0.673,0.631,0.721,0.706,0.036,0.036
26,3,temporada_2015,557.0,0.708,0.649,0.844,0.839,0.060,0.064
27,3,temporada_2016,395.0,0.609,0.627,0.695,0.709,0.075,0.065
33,3,temporada_2022,431.0,0.683,0.704,0.760,0.771,0.058,0.056


In [8]:
cols = ["horizonte", "bloque", "positivos_cand", "razon_mae_cand", "razon_mae_ref", "razon_rmse_cand", "razon_rmse_ref"]
mae_rule["metricas"][cols].round(3)


,horizonte,bloque,positivos_cand,razon_mae_cand,razon_mae_ref,razon_rmse_cand,razon_rmse_ref
3,2,temporada_2010,234.0,0.857,0.907,0.867,0.899
8,2,temporada_2015,557.0,0.860,0.889,0.913,0.932
9,2,temporada_2016,395.0,1.067,1.030,1.029,0.995
15,2,temporada_2022,431.0,0.960,0.946,0.983,0.965
16,2,temporada_2023,1039.0,0.789,0.808,0.859,0.893
17,2,temporada_2024,857.0,1.128,1.136,1.204,1.316
21,3,temporada_2010,234.0,0.828,0.891,0.862,0.890
26,3,temporada_2015,557.0,0.785,0.828,0.880,0.914
27,3,temporada_2016,395.0,1.119,1.042,1.080,1.016
33,3,temporada_2022,431.0,0.959,0.927,1.006,0.990


In [9]:
w = pd.concat({s: e["w"] for s, e in run["evaluaciones"].items()}, names=["semilla"]).reset_index(level=0)
display(w.groupby("horizonte").w.agg(["median", "min", "max"]).round(2))
pd.concat({r: d["lectura_sin_2010"][["horizonte", "veredicto"]] for r, d in (("F1", f1_rule), ("razón MAE", mae_rule))},
          names=["regla"]).reset_index(level=0)


,median,min,max
horizonte,,,
2,0.64,0.58,1.0
3,0.76,0.65,1.0
4,0.81,0.66,1.0


,regla,horizonte,veredicto
0,F1,2,empate
1,F1,3,gana
2,F1,4,gana
0,razón MAE,2,empate
1,razón MAE,3,empate
2,razón MAE,4,empate


## 4 · Salidas: métricas y MLflow


In [10]:
to_json = lambda d: {k: diag.to_json(v) for k, v in d.items() if isinstance(v, pd.DataFrame)}
content = {"enmienda_6a_prima": {
    "guardas": {"G14": g14, "G16_reproduccion": g16_repro},
    "convergencia": {"convergen_sin_penalizar": converged, "ajustes": total, "interpretable": bool(interpretable),
                     "por_bloque": diag.to_json(fits.drop(columns=["corte"]))},
    "lectura_regla_f1": to_json(f1_rule), "lectura_regla_mae": to_json(mae_rule),
    "lectura_regla_mae_mu_sin_mezclar": to_json(mae_rule_mu), "w_5a": diag.to_json(w.drop(columns=["corte"])),
}}
content = json.loads(json.dumps(content, default=lambda o: o.item() if hasattr(o, "item") else str(o)))
for section, value in content.items():
    diag.save_metrics_section(METRICS, section, value)
verdict_text = lambda d: "; ".join(f"h{r.horizonte}: {r.veredicto}" for r in d["veredicto"].itertuples())
tracking.log_study(
    "mejora_modelo_fase6_otras_familias", pd.DataFrame(columns=["numero", "estado"]),
    parent={"params": {"fase": "6a_prima", "preregistro": "e37e55c", "statsmodels": statsmodels.__version__,
                       "convergen": f"{converged}/{total}", "interpretable": str(bool(interpretable)),
                       "f1": verdict_text(f1_rule), "mae": verdict_text(mae_rule)},
            "metricas": {}},
    artifacts=[METRICS], tables={"veredicto_f1": f1_rule["veredicto"], "veredicto_mae": mae_rule["veredicto"]},
    description="Enmienda 6a′ de la fase 6: GLM Poisson con la α de la NB2 (lectura).")


C:\Users\Rosa\RufoEsEternoElRegreso\Denguekay\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


MLflow: estudio b4aaa1cfb6c94074a3dcbb552409afe2 en 'denguekay/mejora_modelo_fase6_otras_familias' (sqlite:///C:/Users/Rosa/RufoEsEternoElRegreso/Denguekay/mlflow.db)


'b4aaa1cfb6c94074a3dcbb552409afe2'